# VitaVision Full Colab Pipeline

This notebook is designed for Google Colab. It rebuilds the VitaVision data science workflow in one file:

1. Upload or locate the project data files.
2. Read and standardize cleaned nutrient CSV files.
3. Clean values and create labels from reference ranges.
4. Build the final labeled dataset.
5. Prepare preprocessing and patient-level splits.
6. Train and compare Logistic Regression and Random Forest models.
7. Save model artifacts and metadata.

Upload a ZIP that contains the `data` folder, or upload the cleaned CSV files directly.

## 1. Install / Import Libraries

In [ ]:
from pathlib import Path
import json
import shutil
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

RANDOM_STATE = 42
LABEL_ORDER = ['Deficient', 'Normal', 'Excessive']
FEATURES = ['Age', 'Gender', 'Nutrient', 'Value']
TARGET = 'Label'
GROUP = 'SEQN'

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

print('Running in Colab:', IN_COLAB)

## 2. Upload Project Data

In Colab, run this cell and upload either:

- A ZIP file containing the VitaVision project or `data` folder.
- The cleaned CSV files directly, such as `vitamin_d_cleaned.csv`, `zinc_cleaned.csv`, and so on.

If you already placed files in Colab or Google Drive, you can skip the upload prompt and set `MANUAL_DATA_DIR` in the next cell.

In [ ]:
UPLOAD_ROOT = Path('/content/vitavision_input') if IN_COLAB else Path.cwd()
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    print('Upload a VitaVision project ZIP, data ZIP, or cleaned CSV files.')
    uploaded = files.upload()
    for filename in uploaded:
        source_path = Path(filename)
        target_path = UPLOAD_ROOT / source_path.name
        if source_path.exists() and source_path.resolve() != target_path.resolve():
            shutil.move(str(source_path), str(target_path))

        if target_path.suffix.lower() == '.zip':
            extract_dir = UPLOAD_ROOT / target_path.stem
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(target_path, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print('Extracted:', extract_dir)
        else:
            print('Uploaded:', target_path)
else:
    print('Local mode. Upload cell skipped.')

## 3. Locate Data Folder

In [ ]:
MANUAL_DATA_DIR = None

CLEANED_FILE_NAMES = [
    'zinc_cleaned.csv', 'vitamin_e_cleaned.csv', 'vitamin_a_cleaned.csv',
    'vitamin_d_cleaned.csv', 'vitamin_c_cleaned.csv', 'magnesium_cleaned.csv',
    'folate_cleaned.csv', 'ferritin_cleaned.csv', 'vitamin_k_cleaned.csv',
    'b12_cleaned.csv', 'b6_cleaned.csv', 'calcium_cleaned.csv',
]

def count_cleaned_files(path: Path) -> int:
    return sum((path / name).exists() for name in CLEANED_FILE_NAMES)

def find_data_dir() -> Path:
    if MANUAL_DATA_DIR:
        manual = Path(MANUAL_DATA_DIR)
        if manual.exists():
            return manual

    roots = [UPLOAD_ROOT, Path.cwd(), Path('/content') if IN_COLAB else Path.cwd()]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        candidates.append(root)
        candidates.extend([path for path in root.rglob('*') if path.is_dir() and path.name.lower() == 'data'])

    scored = sorted(
        [(count_cleaned_files(path), path) for path in candidates],
        key=lambda item: item[0],
        reverse=True,
    )
    if scored and scored[0][0] > 0:
        return scored[0][1]

    raise FileNotFoundError('Could not find cleaned VitaVision CSV files. Upload the data folder or set MANUAL_DATA_DIR.')

DATA_DIR = find_data_dir()
print('DATA_DIR:', DATA_DIR)
print('Cleaned files found:', count_cleaned_files(DATA_DIR), '/', len(CLEANED_FILE_NAMES))

## 4. Data Collection Map

The cleaned source files do not all use the same column names. This map tells the notebook how to standardize every nutrient into the same modeling schema.

In [ ]:
NUTRIENT_SOURCES = [
    {'file': 'zinc_cleaned.csv', 'nutrient': 'Zinc', 'value_col': 'Zinc', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'vitamin_e_cleaned.csv', 'nutrient': 'Vitamin_E', 'value_col': 'LBXVIE', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'vitamin_a_cleaned.csv', 'nutrient': 'Vitamin_A', 'value_col': 'LBXVIA', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'vitamin_d_cleaned.csv', 'nutrient': 'Vitamin_D', 'value_col': 'Vitamin_D', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'vitamin_c_cleaned.csv', 'nutrient': 'Vitamin_C', 'value_col': 'LBXVIC', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'magnesium_cleaned.csv', 'nutrient': 'Magnesium', 'value_col': 'Magnesium', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'folate_cleaned.csv', 'nutrient': 'Folate', 'value_col': 'LBDFOT', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'ferritin_cleaned.csv', 'nutrient': 'Ferritin', 'value_col': 'LBXFER', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'vitamin_k_cleaned.csv', 'nutrient': 'Vitamin_K', 'value_col': 'Vitamin_K', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'b12_cleaned.csv', 'nutrient': 'B12', 'value_col': 'LBDB12', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'b6_cleaned.csv', 'nutrient': 'B6', 'value_col': 'LBXPLP', 'age_col': 'RIDAGEYR', 'gender_col': 'RIAGENDR'},
    {'file': 'calcium_cleaned.csv', 'nutrient': 'Calcium', 'value_col': 'Calcium', 'age_col': 'Age', 'gender_col': 'Gender'},
]

REFERENCE_RANGES = {
    'Zinc': {'low': 66, 'high': 106, 'max_reasonable': 300},
    'Vitamin_E': {'low': 500, 'high': 2000, 'max_reasonable': 5000},
    'Vitamin_A': {'low': 28, 'high': 86, 'max_reasonable': 250},
    'Vitamin_D': {'low': 20, 'high': 50, 'max_reasonable': 200},
    'Vitamin_C': {'low': 0.4, 'high': 2.0, 'max_reasonable': 5},
    'Magnesium': {'low': 1.7, 'high': 2.2, 'max_reasonable': 5},
    'Folate': {'low': 3, 'high': 20, 'max_reasonable': 60},
    'Vitamin_K': {'low': 0.10, 'high': 2.20, 'max_reasonable': 8},
    'B12': {'low': 200, 'high': 900, 'max_reasonable': 3000},
    'B6': {'low': 20, 'high': 100, 'max_reasonable': 400},
    'Calcium': {'low': 8.6, 'high': 10.2, 'max_reasonable': 16},
}

def ferritin_range(gender):
    return {'low': 30, 'high': 400, 'max_reasonable': 1200} if int(gender) == 1 else {'low': 13, 'high': 150, 'max_reasonable': 800}

def reference_range(nutrient, gender):
    if nutrient == 'Ferritin':
        return ferritin_range(gender)
    return REFERENCE_RANGES[nutrient]

## 5. Read, Clean, and Standardize Files

In [ ]:
def read_csv_auto(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=None, engine='python')

def standardize_source(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    required = ['SEQN', config['age_col'], config['gender_col'], config['value_col']]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"{config['file']} is missing columns: {missing}. Found: {list(df.columns)}")

    out = pd.DataFrame({
        'SEQN': df['SEQN'],
        'Age': df[config['age_col']],
        'Gender': df[config['gender_col']],
        'Nutrient': config['nutrient'],
        'Value': df[config['value_col']],
        'Source_File': config['file'],
    })

    for column in ['SEQN', 'Age', 'Gender', 'Value']:
        out[column] = pd.to_numeric(out[column], errors='coerce')

    out['Nutrient'] = out['Nutrient'].astype(str).str.strip()
    out = out.dropna(subset=['SEQN', 'Age', 'Gender', 'Nutrient', 'Value']).copy()
    out = out[(out['Age'].between(0, 120)) & (out['Gender'].isin([1, 2])) & (out['Value'] > 0)]
    out['SEQN'] = out['SEQN'].astype(int)
    out['Gender'] = out['Gender'].astype(int)
    return out

def build_clean_dataset(data_dir: Path) -> pd.DataFrame:
    frames = []
    missing_files = []
    for config in NUTRIENT_SOURCES:
        path = data_dir / config['file']
        if not path.exists():
            missing_files.append(config['file'])
            continue
        source_df = read_csv_auto(path)
        frames.append(standardize_source(source_df, config))

    if missing_files:
        print('Missing files:', missing_files)
    if not frames:
        raise FileNotFoundError('No cleaned nutrient files were loaded.')

    clean_df = pd.concat(frames, ignore_index=True)
    before = len(clean_df)
    clean_df = clean_df.drop_duplicates(subset=['SEQN', 'Age', 'Gender', 'Nutrient', 'Value']).reset_index(drop=True)
    print('Rows before duplicate removal:', before)
    print('Rows after duplicate removal:', len(clean_df))
    return clean_df

clean_df = build_clean_dataset(DATA_DIR)
print('Clean dataset shape:', clean_df.shape)
display(clean_df.head())
display(clean_df['Nutrient'].value_counts().rename_axis('Nutrient').reset_index(name='Rows'))

## 6. Label Data

In [ ]:
DROP_UNREALISTIC_VALUES = True

def label_row(row) -> str:
    info = reference_range(row['Nutrient'], row['Gender'])
    value = float(row['Value'])
    if value <= 0:
        return 'Invalid'
    if DROP_UNREALISTIC_VALUES and value > info['max_reasonable']:
        return 'Invalid'
    if value < info['low']:
        return 'Deficient'
    if value <= info['high']:
        return 'Normal'
    return 'Excessive'

labeled_df = clean_df.copy()
labeled_df['Label'] = labeled_df.apply(label_row, axis=1)

invalid_count = int((labeled_df['Label'] == 'Invalid').sum())
modeling_df = labeled_df[labeled_df['Label'].isin(LABEL_ORDER)].copy()
final_dataset = modeling_df[['SEQN', 'Age', 'Gender', 'Nutrient', 'Value', 'Label']].reset_index(drop=True)

print('Invalid rows removed from modeling:', invalid_count)
print('Final labeled dataset shape:', final_dataset.shape)
display(final_dataset.head())
display(final_dataset['Label'].value_counts().rename_axis('Label').reset_index(name='Rows'))

## 7. Save Final Labeled Dataset

In [ ]:
OUTPUT_DIR = Path('/content/vitavision_outputs') if IN_COLAB else DATA_DIR / 'colab_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_dataset_path = OUTPUT_DIR / 'vitavision_final_labeled_dataset.csv'
final_dataset.to_csv(final_dataset_path, index=False)
print('Saved:', final_dataset_path)

display(final_dataset.groupby(['Nutrient', 'Label']).size().unstack(fill_value=0))

## 8. Prepare Training Dataset

In [ ]:
MODEL_NUTRIENT_NAME_MAP = {
    'Vitamin_D': 'Vitamin D',
    'Vitamin_C': 'Vitamin C',
    'Vitamin_A': 'Vitamin A',
    'Vitamin_E': 'Vitamin E',
    'Vitamin_K': 'Vitamin K',
    'B12': 'Vitamin B12',
    'B6': 'Vitamin B6',
}

df = final_dataset.copy()
df['Nutrient'] = df['Nutrient'].replace(MODEL_NUTRIENT_NAME_MAP)
df['SEQN'] = pd.to_numeric(df['SEQN'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Gender'] = pd.to_numeric(df['Gender'], errors='coerce')
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
df['Label'] = df['Label'].astype(str).str.strip()
df = df.dropna(subset=['SEQN', 'Age', 'Gender', 'Nutrient', 'Value', 'Label']).copy()
df = df[df['Label'].isin(LABEL_ORDER)].reset_index(drop=True)
df['SEQN'] = df['SEQN'].astype(int)
df['Gender'] = df['Gender'].astype(int)

print('Training dataset shape:', df.shape)
print('Patients:', df['SEQN'].nunique())
display(df.head())

## 9. Patient-Level Train / Validation / Test Split

In [ ]:
def patient_level_split(data: pd.DataFrame):
    first_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    train_val_idx, test_idx = next(first_splitter.split(data, data[TARGET], groups=data[GROUP]))

    train_val_df = data.iloc[train_val_idx].reset_index(drop=True)
    test_df = data.iloc[test_idx].reset_index(drop=True)

    second_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    train_idx, val_idx = next(second_splitter.split(train_val_df, train_val_df[TARGET], groups=train_val_df[GROUP]))

    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = patient_level_split(df)
split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'patients': [train_df[GROUP].nunique(), val_df[GROUP].nunique(), test_df[GROUP].nunique()],
})
split_summary['row_percent'] = (split_summary['rows'] / len(df) * 100).round(2)
display(split_summary)

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

## 10. Build Models

In [ ]:
def build_logistic_regression_model() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), ['Nutrient']),
            ('num', StandardScaler(), ['Age', 'Gender', 'Value']),
        ]
    )
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE, class_weight='balanced')),
    ])

def build_random_forest_model() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), ['Nutrient']),
            ('num', 'passthrough', ['Age', 'Gender', 'Value']),
        ]
    )
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)),
    ])

baseline_model = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
logistic_model = build_logistic_regression_model()
random_forest_model = build_random_forest_model()

## 11. Train and Evaluate

In [ ]:
def score_predictions(split_name: str, y_true, y_pred) -> dict:
    return {
        'split': split_name,
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
    }

baseline_model.fit(X_train, y_train)
logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

baseline_val_pred = baseline_model.predict(X_val)
logistic_val_pred = logistic_model.predict(X_val)
logistic_test_pred = logistic_model.predict(X_test)
rf_val_pred = random_forest_model.predict(X_val)
rf_test_pred = random_forest_model.predict(X_test)

scores = [
    score_predictions('validation_baseline', y_val, baseline_val_pred),
    score_predictions('validation_logistic_regression', y_val, logistic_val_pred),
    score_predictions('test_logistic_regression', y_test, logistic_test_pred),
    score_predictions('validation_random_forest', y_val, rf_val_pred),
    score_predictions('test_random_forest', y_test, rf_test_pred),
]
comparison_df = pd.DataFrame(scores)
display(comparison_df)

print('Logistic Regression validation report')
print(classification_report(y_val, logistic_val_pred, digits=4))
print('Random Forest validation report')
print(classification_report(y_val, rf_val_pred, digits=4))
print('Random Forest test report')
print(classification_report(y_test, rf_test_pred, digits=4))

display(pd.DataFrame(confusion_matrix(y_test, rf_test_pred, labels=LABEL_ORDER), index=LABEL_ORDER, columns=LABEL_ORDER))

## 12. Select Best Model

In [ ]:
model_candidates = [
    {
        'name': 'Logistic Regression',
        'model': logistic_model,
        'validation': comparison_df[comparison_df['split'] == 'validation_logistic_regression'].iloc[0].to_dict(),
        'test': comparison_df[comparison_df['split'] == 'test_logistic_regression'].iloc[0].to_dict(),
    },
    {
        'name': 'Random Forest',
        'model': random_forest_model,
        'validation': comparison_df[comparison_df['split'] == 'validation_random_forest'].iloc[0].to_dict(),
        'test': comparison_df[comparison_df['split'] == 'test_random_forest'].iloc[0].to_dict(),
    },
]

best_candidate = max(model_candidates, key=lambda item: item['validation']['macro_f1'])
print('Selected model:', best_candidate['name'])
print('Validation macro F1:', best_candidate['validation']['macro_f1'])
print('Test macro F1:', best_candidate['test']['macro_f1'])

## 13. Save Model Artifacts

In [ ]:
MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

logistic_model_path = MODEL_DIR / 'vitavision_logistic_regression_model.pkl'
random_forest_model_path = MODEL_DIR / 'vitavision_random_forest_model.pkl'
unified_model_path = MODEL_DIR / 'vitavision_unified_model.pkl'
metadata_path = MODEL_DIR / 'vitavision_unified_model_metadata.json'
comparison_path = MODEL_DIR / 'vitavision_model_comparison.csv'

joblib.dump(logistic_model, logistic_model_path)
joblib.dump(random_forest_model, random_forest_model_path)
joblib.dump(best_candidate['model'], unified_model_path)
comparison_df.to_csv(comparison_path, index=False)

metadata = {
    'model_name': 'VitaVision Unified Model',
    'selected_model': best_candidate['name'],
    'model_file': str(unified_model_path),
    'logistic_regression_model_file': str(logistic_model_path),
    'random_forest_model_file': str(random_forest_model_path),
    'model_comparison_file': str(comparison_path),
    'final_dataset_file': str(final_dataset_path),
    'features': FEATURES,
    'target': TARGET,
    'classes': LABEL_ORDER,
    'random_state': RANDOM_STATE,
    'split_strategy': 'patient-level GroupShuffleSplit using SEQN',
    'train_rows': int(len(train_df)),
    'validation_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
    'validation_accuracy': float(best_candidate['validation']['accuracy']),
    'validation_macro_f1': float(best_candidate['validation']['macro_f1']),
    'test_accuracy': float(best_candidate['test']['accuracy']),
    'test_macro_f1': float(best_candidate['test']['macro_f1']),
    'nutrient_name_convention': 'Model uses names such as Vitamin D, Vitamin B12, Zinc, Ferritin',
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Saved artifacts to:', MODEL_DIR)
print('Unified model:', unified_model_path)
print('Metadata:', metadata_path)
print('Comparison:', comparison_path)

## 14. Test a New Patient Example

In [ ]:
sample_patient = pd.DataFrame([
    {'Age': 25, 'Gender': 1, 'Nutrient': 'Vitamin D', 'Value': 12.0},
    {'Age': 25, 'Gender': 1, 'Nutrient': 'Vitamin B12', 'Value': 520.0},
    {'Age': 25, 'Gender': 1, 'Nutrient': 'Zinc', 'Value': 120.0},
])

sample_patient['Prediction'] = best_candidate['model'].predict(sample_patient[FEATURES])
if hasattr(best_candidate['model'], 'predict_proba'):
    sample_patient['Confidence'] = [f'{max(row) * 100:.1f}%' for row in best_candidate['model'].predict_proba(sample_patient[FEATURES])]

display(sample_patient)

## 15. Download Outputs from Colab

In [ ]:
archive_base = OUTPUT_DIR.parent / 'vitavision_colab_outputs'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_DIR)
print('Created archive:', archive_path)

if IN_COLAB:
    files.download(archive_path)